In [27]:
import unicodedata
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def remove_accents(text):#去除西语名字音调
    if pd.isna(text):
        return text
    text = str(text)
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text

def ip_to_outs(IP):
    whole = int(IP)
    frac = round(IP - whole, 1)
    return whole * 3 + int(frac * 10)

def outs_to_ip(outs):
    return outs // 3 + (outs % 3) / 10


df = pd.read_excel('pitchers2025regular.xlsx')
df = df.rename(columns={'WAR': 'rWAR'})
df_fwar = pd.read_excel('fangraphWar25.xlsx')
df_fwar = df_fwar.rename(columns={'Name': 'Player'})
df_fwar = df_fwar.rename(columns={'Pit WAR': 'fWAR'})
df['Player'] = df['Player'].str.replace('*', '', regex=False)#去掉名字后面的*

df['outs'] = df['IP'].apply(ip_to_outs)

df_fwar['IP'] = df_fwar['IP'].fillna(0)
df_fwar['outs'] = df_fwar['IP'].apply(ip_to_outs)

df['K_Per'] = (df['SO'] / df['BF'] * 100).round(1)
df['BB_Per'] = (df['BB'] / df['BF'] * 100).round(1)

df['Role'] = 0
df.loc[df['outs'] >= 140*3, 'Role'] = 1
df_fwar['Role'] = 0
df_fwar.loc[df_fwar['outs'] >= 140*3, 'Role'] = 1

In [ ]:
df['player_norm'] = df['Player'].apply(remove_accents)
df_fwar['player_norm'] = df_fwar['Player'].apply(remove_accents)


df = df.merge(
    df_fwar[['player_norm','Role','fWAR']],
    on=['player_norm','Role'], 
    how='left'
)

df = df.drop(columns=['Player-additional', 'Awards', 'player_norm','W-L%'],errors= 'ignore')

df['WAR_dif'] = df['fWAR']-df['rWAR']

df['HR_Per'] = (((df['HR']) / df['H']) * 100).round(2)

df[df['Role'] == 1].sort_values(by='HR_Per', ascending=False)


,Rk,Player,Age,Team,Lg,rWAR,W,L,ERA,G,...,BB9,SO9,SO/BB,outs,K_Per,BB_Per,Role,fWAR,WAR_dif,HR_Per
76,77,Shota Imanaga,31,CHC,NL,1.5,9,8,3.73,25,...,1.6,7.3,4.50,434,20.6,4.6,1,0.9,-0.6,26.50
28,29,Jacob deGrom,37,TEX,AL,2.9,12,8,2.97,30,...,1.9,9.6,5.00,518,27.7,5.5,1,3.4,0.5,21.31
11,12,Zack Littell,29,2TM,2LG,3.2,10,8,3.81,32,...,1.5,6.3,4.06,560,17.1,4.2,1,1.5,-1.7,20.69
74,75,Seth Lugo,35,KCR,AL,1.7,8,7,4.15,26,...,3.4,7.7,2.27,436,20.5,9.0,1,0.5,-1.2,20.30
20,21,Jake Irvin,28,WSN,NL,-0.4,9,13,5.70,33,...,3.1,6.2,2.00,540,15.8,7.9,1,-0.4,0.0,19.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,10,Paul Skenes,23,PIT,NL,7.7,10,10,1.97,32,...,2.0,10.4,5.14,563,29.5,5.7,1,6.5,-1.2,8.09
35,36,José Soriano,26,LAA,AL,2.4,10,11,4.26,31,...,4.2,8.1,1.95,507,21.0,10.8,1,3.0,0.6,7.59
2,3,Cristopher Sánchez,28,PHI,NL,8.0,13,5,2.50,32,...,2.0,9.4,4.82,606,26.3,5.5,1,6.4,-1.6,7.02
0,1,Logan Webb,28,SFG,NL,3.8,15,11,3.22,34,...,2.0,9.7,4.87,621,26.2,5.4,1,5.5,1.7,6.67


In [58]:
df_starters = df[df['Role'] == 1].copy()

df_starters['HR_Per'].corr(df_starters['FIP'])
df_starters['HR_Per'].corr(df_starters['fWAR'])

np.float64(-0.4752749477558946)

In [52]:
df.loc[
    (df['Role'] == 1) & (df['WAR_dif'] > 1)
]['ERA'].mean()

np.float64(4.606086956521739)

In [53]:
df.loc[
    (df['Role'] == 1) & (df['WAR_dif'] < 0.5)
]['ERA'].mean()

np.float64(3.6461224489795923)

In [ ]:
#df_starter.plot(x='', y='WAR_dif', kind='scatter')

df_numeric = df.select_dtypes(include='number')
target_col = 'WAR_dif'  # 目标列

# 1. 计算所有列与目标列的 Pearson 相关系数
corrs = df_numeric.corr()[target_col]


# 2. 去掉自己（目标列本身的相关系数是 1）
corrs = corrs.drop(target_col)

# 3. 找绝对值最大
strongest_col = corrs.abs().idxmax()
strongest_value = corrs[strongest_col]

corrs_sorted = corrs.abs().sort_values(ascending=False)
print(corrs_sorted)

print(f"与 {target_col} 线性关系最强的列是 {strongest_col}，相关系数为 {strongest_value:.3f}")

corrs_sorted.plot(kind='bar', figsize=(10,6))
plt.ylabel('Correlation with FIP')
plt.title('Correlation of all numeric columns with FIP')
plt.show()

In [ ]:
df.loc[:,['Player','HR_Per']]
df[df['Role'] == 1].sort_values(by='WAR_dif', ascending=False)
df['WAR_dif'].corr(df['HR_Per'])


df.loc[[78, 39]]
